# 01 - Data Understanding
**Proyek:** Hotel Booking Cancellation Prediction  |  **Tahap:** M2  |  **PRD:** v1.1

Tujuan notebook ini: memahami dataset sebelum EDA dan modeling.
Isi: info dataset, kamus kolom, missing value, duplikat, nilai tidak wajar, distribusi target, dan cek leakage.

**Sumber dataset:** Hotel Booking Demand (Kaggle, `jessemostipak/hotel-booking-demand`). Publikasi: Antonio, de Almeida & Nunes (2019), Data in Brief 22, 41-49, https://doi.org/10.1016/j.dib.2018.11.126

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

# Cari file CSV otomatis (Kaggle / Colab / lokal)
candidates = (glob.glob('/kaggle/input/**/hotel_bookings.csv', recursive=True)
              + glob.glob('./hotel_bookings.csv') + glob.glob('./data/raw/hotel_bookings.csv')
              + glob.glob('/mnt/user-data/uploads/hotel_bookings.csv'))
assert candidates, 'hotel_bookings.csv tidak ditemukan, atur DATA_PATH manual.'
DATA_PATH = candidates[0]
print('File:', DATA_PATH)
print('Ukuran file: %.2f MB' % (os.path.getsize(DATA_PATH)/1e6))

## 1. Memuat data

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Jumlah baris, kolom:', df.shape)
df.head()

## 2. Tipe data dan kamus kolom

In [ ]:
df.info()

In [ ]:
# Kamus kolom (ringkas, mengacu dokumentasi dataset)
kamus = {
 'hotel': 'Jenis hotel: City Hotel atau Resort Hotel',
 'is_canceled': 'TARGET. 1 = dibatalkan, 0 = tidak dibatalkan',
 'lead_time': 'Jumlah hari antara tanggal pemesanan dan tanggal kedatangan',
 'arrival_date_year': 'Tahun kedatangan',
 'arrival_date_month': 'Bulan kedatangan',
 'arrival_date_week_number': 'Nomor minggu kedatangan dalam tahun',
 'arrival_date_day_of_month': 'Tanggal kedatangan',
 'stays_in_weekend_nights': 'Jumlah malam akhir pekan (Sabtu/Minggu) menginap',
 'stays_in_week_nights': 'Jumlah malam hari kerja (Senin-Jumat) menginap',
 'adults': 'Jumlah dewasa',
 'children': 'Jumlah anak',
 'babies': 'Jumlah bayi',
 'meal': 'Jenis paket makan (BB, HB, FB, SC, Undefined)',
 'country': 'Negara asal tamu (kode ISO)',
 'market_segment': 'Segmen pasar / asal pemesanan',
 'distribution_channel': 'Kanal distribusi pemesanan',
 'is_repeated_guest': '1 jika tamu pernah menginap sebelumnya',
 'previous_cancellations': 'Jumlah pembatalan sebelumnya oleh tamu',
 'previous_bookings_not_canceled': 'Jumlah pemesanan sebelumnya yang tidak dibatalkan',
 'reserved_room_type': 'Tipe kamar yang dipesan (kode)',
 'assigned_room_type': 'Tipe kamar yang diberikan (kode)',
 'booking_changes': 'Jumlah perubahan pada pemesanan',
 'deposit_type': 'Jenis deposit: No Deposit, Non Refund, Refundable',
 'agent': 'ID agen travel (jika ada)',
 'company': 'ID perusahaan pemesan (jika ada)',
 'days_in_waiting_list': 'Hari di daftar tunggu sebelum dikonfirmasi',
 'customer_type': 'Tipe pelanggan: Transient, Contract, Group, Transient-party',
 'adr': 'Average Daily Rate (rata-rata tarif harian)',
 'required_car_parking_spaces': 'Jumlah slot parkir yang diminta',
 'total_of_special_requests': 'Jumlah permintaan khusus',
 'reservation_status': 'Status akhir reservasi (Canceled, Check-Out, No-Show) -> LEAKAGE',
 'reservation_status_date': 'Tanggal status terakhir diperbarui -> LEAKAGE',
}
info = pd.DataFrame({'tipe': df.dtypes.astype(str), 'n_unik': df.nunique(),
                     'arti': pd.Series(kamus)})
info

## 3. Missing value

In [ ]:
miss = df.isna().sum()
miss_df = pd.DataFrame({'jumlah': miss, 'persen': (miss/len(df)*100).round(2)})
miss_df = miss_df[miss_df.jumlah > 0].sort_values('jumlah', ascending=False)
miss_df

**Catatan:** `children` (4) dan `country` (488) hanya sedikit. `agent` dan `company` kosong berarti *tidak ada agen/perusahaan*, bukan data hilang acak. `company` 94% kosong sehingga tidak dipakai sebagai fitur.

## 4. Duplikat

In [ ]:
cols_dup = [c for c in df.columns if c != 'reservation_status_date']
dup_all = df.duplicated().sum()
dup_cols = df.duplicated(subset=cols_dup).sum()
print('Duplikat (semua kolom)                 :', dup_all)
print('Duplikat (tanpa reservation_status_date):', dup_cols, '(%.1f%%)' % (dup_cols/len(df)*100))
df_dedup = df.drop_duplicates(subset=cols_dup).reset_index(drop=True)
print('Baris setelah duplikat dibuang          :', len(df_dedup))

**Keputusan (PRD 7.5):** duplikat dihapus **sebelum split** agar baris identik tidak masuk train dan test sekaligus (risiko leakage). Hasil sebelum vs sesudah dilaporkan nanti.

## 5. Distribusi target

In [ ]:
tgt = df.is_canceled.value_counts().rename('jumlah').to_frame()
tgt['persen'] = (tgt.jumlah/len(df)*100).round(2)
display(tgt)

tgt2 = df_dedup.is_canceled.value_counts(normalize=True).mul(100).round(2)
print('Setelah duplikat dibuang (%):'); print(tgt2.to_string())

fig, ax = plt.subplots(1, 2, figsize=(9,3.5))
for a, d, t in zip(ax, [df, df_dedup], ['Data asli', 'Tanpa duplikat']):
    sns.countplot(x='is_canceled', data=d, ax=a); a.set_title(t)
plt.tight_layout(); plt.show()

## 6. Nilai tidak wajar

In [ ]:
guests = df.adults + df.children.fillna(0) + df.babies
nights = df.stays_in_weekend_nights + df.stays_in_week_nights
anom = pd.Series({
 'tamu = 0': int((guests == 0).sum()),
 'malam menginap = 0': int((nights == 0).sum()),
 'adr < 0': int((df.adr < 0).sum()),
 'adr = 0': int((df.adr == 0).sum()),
 'adr > 1000': int((df.adr > 1000).sum()),
 'adults > 10': int((df.adults > 10).sum()),
 'lead_time > 365': int((df.lead_time > 365).sum()),
})
display(anom.to_frame('jumlah baris'))
df[['lead_time','adults','children','babies','adr']].describe().T

**Catatan:** nilai tidak wajar dicatat dulu. Keputusan dibuang/dibatasi ditentukan di tahap EDA/preprocessing beserta dampaknya, tidak dihapus di sini.

## 7. Cek leakage dan hubungan awal dengan target

In [ ]:
print(pd.crosstab(df.reservation_status, df.is_canceled))

`Canceled` dan `No-Show` selalu bernilai `is_canceled = 1`, dan `Check-Out` selalu 0. Jadi **`reservation_status` dan `reservation_status_date` dikeluarkan dari fitur**.

In [ ]:
for col in ['hotel', 'deposit_type', 'market_segment', 'customer_type']:
    t = (df.groupby(col).is_canceled.agg(['count','mean']).rename(columns={'mean':'rasio_batal'})
         .sort_values('count', ascending=False))
    t['rasio_batal'] = t.rasio_batal.round(3)
    print('\n==', col); display(t)

**Catatan:** `deposit_type = Non Refund` memiliki rasio batal sangat tinggi. Ini pola janggal pada dataset, dicatat sebagai keterbatasan dan bukan klaim sebab-akibat.

## 8. Ringkasan audit

In [ ]:
ringkasan = pd.Series({
 'baris': len(df), 'kolom': df.shape[1],
 'rasio_batal_%': round(df.is_canceled.mean()*100, 2),
 'duplikat': int(dup_cols),
 'baris_setelah_dedup': len(df_dedup),
 'rasio_batal_dedup_%': round(df_dedup.is_canceled.mean()*100, 2),
 'missing_children': int(df.children.isna().sum()),
 'missing_country': int(df.country.isna().sum()),
 'missing_agent': int(df.agent.isna().sum()),
 'missing_company': int(df.company.isna().sum()),
 'tamu_nol': int((guests==0).sum()),
 'malam_nol': int((nights==0).sum()),
})
display(ringkasan.to_frame('nilai'))
os.makedirs('reports', exist_ok=True)
ringkasan.to_csv('reports/data_audit_summary.csv', header=['nilai'])

### Kesimpulan M2
1. Target `is_canceled` imbalance ringan, bukan berat.
2. Duplikat besar (~27%) dan akan dihapus sebelum split.
3. `reservation_status` dan `reservation_status_date` adalah leakage dan tidak dipakai.
4. `company` terlalu banyak kosong; `agent`/`company` kosong bermakna "tidak ada".
5. Nilai tidak wajar dicatat dan diputuskan di EDA/preprocessing.

**Langkah berikutnya: M3 EDA** (distribusi fitur, hubungan fitur dengan target, korelasi, outlier, disertai interpretasi tiap visualisasi).